In [1]:

import numpy as np
import pandas as pd
import optuna
import joblib

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances
)

from lab2_utils import get_data, get_split, cv_macro_f1, CLASS_NAMES


# 1. Load the ORIGINAL baseline data (one-hot, no engineered features)
X, y = get_data()

# 2. Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = get_split(X, y)

# 3. Stratified 5-fold cross-validation
cv5 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Dataset:", X.shape)
print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


# 4. Evaluate the original default baseline
baseline = XGBClassifier(
    random_state=42,
    n_jobs=-1
)

baseline_cv = cv_macro_f1(
    baseline, X_train, y_train,
    cv=cv5,
    weights=False
)

print("\nORIGINAL BASELINE")
print(f"Mean CV Macro-F1: {baseline_cv.mean():.4f}")
print(f"Std CV Macro-F1: {baseline_cv.std():.4f}")


# 5. Define the Optuna objective
def objective(trial):

    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),
        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 5
        ),
        "gamma": trial.suggest_float("gamma", 0.0, 1.0),
        "random_state": 42,
        "n_jobs": -1
    }

    model = XGBClassifier(**params)

    scores = cv_macro_f1(
        model,
        X_train,
        y_train,
        cv=cv5,
        weights=False
    )

    return scores.mean()


# 6. Run Bayesian optimization with TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

sampler = optuna.samplers.TPESampler(seed=42)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler
)

study.optimize(
    objective,
    n_trials=30,
    show_progress_bar=False
)

print("\nOPTUNA RESULTS")
print(f"Best CV Macro-F1: {study.best_value:.4f}")
print("\nBest hyperparameters:")

for name, value in study.best_params.items():
    print(f"{name}: {value}")


# 7. Train the optimized model on the entire training set
best_model = XGBClassifier(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train)


# 8. Evaluate on the untouched test set
y_test_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(
    y_test, y_test_pred, average="macro", zero_division=0
)
test_recall = recall_score(
    y_test, y_test_pred, average="macro", zero_division=0
)
test_macro_f1 = f1_score(
    y_test, y_test_pred, average="macro", zero_division=0
)

print("\nOPTIMIZED BASELINE - TEST RESULTS")
print(f"Accuracy:        {test_accuracy:.4f}")
print(f"Macro Precision: {test_precision:.4f}")
print(f"Macro Recall:    {test_recall:.4f}")
print(f"Test Macro-F1:   {test_macro_f1:.4f}")

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_test,
    y_test_pred,
    labels=range(len(CLASS_NAMES)),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))


# 9. Compare original and optimized baseline
comparison = pd.DataFrame({
    "Model": ["Original baseline", "Optimized baseline"],
    "CV Macro-F1": [
        baseline_cv.mean(),
        study.best_value
    ],
    "Test Macro-F1": [
        f1_score(y_test, baseline.fit(X_train, y_train).predict(X_test),
                 average="macro"),
        test_macro_f1
    ]
})

display(comparison.round(4))


# 10. Save the Optuna study and optimized model
joblib.dump(study, "optuna_diana_baseline_study.pkl")
joblib.dump(best_model, "xgb_diana_baseline_optimized.pkl")

print("\nOptuna study and optimized model saved successfully.")


# 11. Optuna optimization plots
plot_optimization_history(study).show()
plot_param_importances(study).show()


Dataset: (2930, 333)
Training samples: 2344
Test samples: 586

ORIGINAL BASELINE
Mean CV Macro-F1: 0.7225
Std CV Macro-F1: 0.0196

OPTUNA RESULTS
Best CV Macro-F1: 0.7452

Best hyperparameters:
max_depth: 6
learning_rate: 0.16865924671225896
n_estimators: 287
subsample: 0.834209000248802
colsample_bytree: 0.6726966831290326
min_child_weight: 1
gamma: 0.24170427208769074

OPTIMIZED BASELINE - TEST RESULTS
Accuracy:        0.7679
Macro Precision: 0.7744
Macro Recall:    0.7363
Test Macro-F1:   0.7522

CLASSIFICATION REPORT
              precision    recall  f1-score   support

       <100k     0.7609    0.7292    0.7447        48
    100–130k     0.6900    0.6765    0.6832       102
    130–180k     0.7565    0.8406    0.7963       207
    180–250k     0.8220    0.7080    0.7608       137
    250–400k     0.8171    0.8481    0.8323        79
       ≥400k     0.8000    0.6154    0.6957        13

    accuracy                         0.7679       586
   macro avg     0.7744    0.7363    0.

,Model,CV Macro-F1,Test Macro-F1
0,Original baseline,0.7225,0.7476
1,Optimized baseline,0.7452,0.7522



Optuna study and optimized model saved successfully.
